# Nettoyage et merge des bases de données communales

Merge les bases sur `codgeo` et `an`.
La base finale a une ligne par couple **(commune × année)**.

## 1. Configuration

In [1]:
import pandas as pd
import os
import numpy as np

base_path = "/Users/margo/Documents/Margot/0.Polytechnique/Statistical-machine-learning/Projet-final/Data/"
caractéristiques_communes_path = os.path.join(base_path, "Caractéristiques-générales-communes")
dvf_path = os.path.join(base_path, "Indicateurs-immobiliers")
comptes_path = os.path.join(base_path, "Comptes-communes") 
obs_path = os.path.join(base_path, "Observatoire-territoires")


## 2. Caractéristiques socio-économiques des communes

#### 2.1 Base de données contenant toutes les caractéristiques générales des communes
Contient : région, département, EPCI, code postal, superficie, densité, coordonnées, grille de densité, polygone...

Base **transversale** (pas d'année). 

**Colonnes conservées :** on garde les variables analytiquement utiles et on exclut les colonnes purement nominatives (URL, noms déclinés) et le polygone (trop lourd pour un DataFrame plat) => même si ça pourrait être utile pour faire des représentations graphiques

In [2]:
geo_path = os.path.join(caractéristiques_communes_path, "communes_france_avec_polygon_2024.csv")
geo = pd.read_csv(geo_path, low_memory=False)

geo = geo.rename(columns={'code_insee': 'codgeo'})
geo['codgeo'] = geo['codgeo'].astype(str).str.zfill(5)

# Colonnes utiles (on exclut polygone, URLs, noms déclinés redondants)
geo_cols_keep = [
    'codgeo',
    'typecom',           # type de commune (COM, COMA, ...)
    'reg_code', 'reg_nom',
    'dep_code', 'dep_nom',
    'epci_code', 'epci_nom',
    'code_postal',
    'zone_emploi',
    'population',
    'superficie_km2',
    'densite',
    'altitude_moyenne',
    'latitude_centre', 'longitude_centre',
    'grille_densite',   # catégorie urbain/rural
]
geo = geo[geo_cols_keep].drop_duplicates('codgeo')

# Préfixer pour clarté (sauf codgeo)
geo = geo.rename(columns={c: f"geo_{c}" for c in geo_cols_keep if c != 'codgeo'})

#### 2.2 Ouverture des bases socio-démographiques
**Structure commune à tous les fichiers :**
- Lignes 0–3 : métadonnées ANCT (à ignorer)
- Ligne 4 : en-têtes réels (`codgeo`, `libgeo`, `an`, + variable)
- Lignes 5+ : données

Certaines bases de données utilisent pour le format d'années un intervalle. On retient l'année de **fin** de la période comme clé de merge.

In [3]:
def normalize_year(val):
    """
    Normalise une valeur d'année en entier.
    '1968-1975' → 1975  (année de fin de l'intervalle)
    2022.0      → 2022
    2021        → 2021
    """
    if pd.isna(val):
        return pd.NA
    s = str(val).strip()
    if '-' in s and len(s) > 5:  # format intervalle ex: '1968-1975'
        return int(s.split('-')[1])
    return int(float(s))


def load_obs_file(filepath):
    """
    Charge un fichier Excel de l'Observatoire des territoires.
    - Ignore les 4 premières lignes de métadonnées (header=4)
    - Formate codgeo en str avec zéro initial (ex: 01001)
    - Normalise la colonne 'an' en entier
    """
    df = pd.read_excel(filepath, header=4)
    df['codgeo'] = df['codgeo'].astype(str).str.strip().str.zfill(5)
    if 'an' in df.columns:
        df['an'] = df['an'].apply(normalize_year)
        df = df.dropna(subset=['an'])
        df['an'] = df['an'].astype(int)
    return df

In [4]:
# Charger toutes les bases de données de l'Observatoire des territoires (caractéristiques socio-économiques))
obs_data = {}
for file in sorted(os.listdir(obs_path)):
    if file.endswith(".xlsx"):
        name = file.replace(".xlsx", "")
        filepath = os.path.join(obs_path, file)
        obs_data[name] = load_obs_file(filepath)

print(f"✓ {len(obs_data)} bases chargées")

✓ 32 bases chargées


#### 2.3 Aperçu des données : années disponibles par base et échelle géographique
Vue d'ensemble de toutes les bases : échelle géographique et années présentes.


In [5]:
# --- RÉCAPITULATIF DÉTAILLÉ DES BASES ---

# On ajuste la largeur pour accommoder la liste des variables
print(f"{'Base':<40} {'Échelle':<12} {'Années':<25} {'Variables (Colonnes)'}")
print("-" * 140)

for name, df in obs_data.items():
    # Détection de l'échelle (on réutilise ta logique sur le fichier raw pour être sûr)
    raw = pd.read_excel(os.path.join(obs_path, [f for f in os.listdir(obs_path) if name in f][0]), header=4)
    max_len = raw['codgeo'].astype(str).str.strip().dropna().apply(len).max()
    echelle = 'Département' if max_len <= 3 else 'Commune'

    # Gestion de l'affichage des années
    if 'an' in df.columns:
        annees = sorted(df['an'].dropna().unique())
        # On raccourcit l'affichage si trop d'années (ex: [2016...2023])
        if len(annees) > 4:
            annees_str = f"[{annees[0]}...{annees[-1]}] ({len(annees)} ans)"
        else:
            annees_str = str(annees)
    else:
        annees_str = '[statique]'

    # Extraction des noms de variables (on exclut les IDs)
    cols_vars = [c for c in df.columns if c not in ['codgeo', 'libgeo', 'an']]
    vars_str = ", ".join(cols_vars)

    # Nettoyage du nom de la base pour l'affichage
    short_name = '_'.join(name.split('_')[1:]) if '_' in name else name
    
    print(f"{short_name:<40} {echelle:<12} {annees_str:<25} {vars_str}")

print("-" * 140)

Base                                     Échelle      Années                    Variables (Colonnes)
--------------------------------------------------------------------------------------------------------------------------------------------
ruralité_revitalisation_FRR             Commune      [statique]                codefrr
aah                                      Commune      [np.int64(2021)]          benef_aah
logt_ALF                                 Commune      [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)] part_alf
logt_ALS                                 Commune      [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)] nb_als
logt_APL                                 Commune      [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)] part_apl
aux_communes                             Commune      [statique]                zonage_afr22
attraction_ville                         Commune      [statique]                aav2020, aav2020_libgeo
A

#### 2.4 Modification et merge des datasets
Il y a une base de données qui n'est pas à l'échelle de la commune mais du département (Nombre de bénéficiaires de l'allocation personnalisée d'autonomie (APA) `benef_apa`). La solution utilisée, attribuer la même valeur aux communes du même département.

Pour les variables qui ne sont pas présentes toutes les années, gérer au cas par cas : 
- Ceratines variables peuvent être constantes au cours du temps :
    - AFR - Aide à finalité régionale (2022-2027) : Communes éligibles `zonage_afr22` => étendue sur les années 2022 à 2025
    - Aires d'attraction des villes 2020 `aav2020` et `aav2020_libego`=> étendue sur les années 2020 à 2025
    - Territoires d'industrie (TI) : Communes bénéficiaires (2023-2027) `ti_zpp2`=> étendue sur les années 2023 à 2025
    - FRR - France Ruralités Revitalisation `codefrr` => étendue sur les années 2024 à 2025
    - Niveau de centres d'équipements et de services des communes `nivcentr` => étendue sur les années 2021 à 2025
    - Taux d'évolution annuel de la population `tx_var_pop` => étendue sur les années 2016 à 2025
    - Villages d'Avenir : Communes bénéficiaires `va_zpp`=> étendue sur les années 2023 à 2025
    - Petites villes de demain (PVD) : Communes bénéficiaires `pvd_zpp` => étendue sur les années 2020 à 2025
    
- Certaines variables ne peuvent pas être mises constantes au cours du temps (par exemple celle concernant les bénéficiaires de l'allocation aux adultes handicapés (AAH) `benef_aah`). Il y aura donc des données manquantes pour certaines années. 

Pour toutes les bases sans années, ajouter une colonne année de 2016 à 2023 et on met constant la variable. Pour les bases à l'échelle du départment, on met la même valeur à toutes les communes d'un même dépatrment. 

À la fin on merge toutes les bases de données de sorte qu'une ligne = commune X année. On utilise : `codgeo` + `an`

In [6]:
# --- 1. CONFIGURATION DES RÈGLES SPÉCIFIQUES ---
annees_projet = list(range(2016, 2024)) # Ta fenêtre d'étude

# Dictionnaire de configuration : {nom_colonne: [liste_annees_extension]}
regles_temporelles = {
    'zonage_afr22': range(2022, 2026),
    'aav2020': range(2020, 2026),
    'aav2020_libgeo': range(2020, 2026),
    'ti_zpp2': range(2023, 2026),
    'codefrr': range(2024, 2026),
    'nivcentr': range(2021, 2026),
    'tx_var_pop': range(2016, 2026),
    'va_zpp': range(2023, 2026),
    'pvd_zpp': range(2020, 2026)
}

def get_variable_cols(df):
    """
    Identifie les colonnes qui ne sont pas des identifiants techniques.
    Cela permet d'isoler uniquement les données (ex: taux de pauvreté, revenus).
    """
    # On exclut les colonnes de gestion géographique et temporelle
    cols_to_exclude = ['codgeo', 'libgeo', 'an', 'code_dept']
    return [c for c in df.columns if c not in cols_to_exclude]

# --- 2. TRAITEMENT DE CHAQUE BASE ---
processed_dfs = []

for name, df in obs_data.items():
    max_len = df['codgeo'].str.len().max()
    is_dept = (max_len <= 3)
    temp_df = df.copy()
    
    var_cols = get_variable_cols(temp_df)
    
    # A. GESTION DE L'EXPANSION TEMPORELLE SPÉCIFIQUE
    # Cas 1 : La base n'a pas de colonne 'an'
    if 'an' not in temp_df.columns:
        expanded_list = []
        # On vérifie si la variable principale de ce fichier est dans nos règles
        # (On prend la première colonne de données comme référence)
        main_var = var_cols[0] if var_cols else None
        
        if main_var in regles_temporelles:
            target_years = regles_temporelles[main_var]
            print(f"Expansion spécifique pour {main_var} sur {list(target_years)}")
        else:
            # Si pas de règle et pas d'année, on considère que c'est constant sur tout le projet
            target_years = annees_projet
            print(f"Expansion totale par défaut pour {name}")
            
        for annee in target_years:
            df_year = temp_df.copy()
            df_year['an'] = annee
            expanded_list.append(df_year)
        temp_df = pd.concat(expanded_list, ignore_index=True)

    # Cas 2 : La base A déjà une colonne 'an' mais on veut peut-être l'étendre (ex: tx_var_pop)
    else:
        main_var = var_cols[0] if var_cols else None
        if main_var in regles_temporelles:
            # On récupère les données existantes et on les duplique pour les années manquantes de la règle
            existing_years = temp_df['an'].unique()
            target_years = regles_temporelles[main_var]
            missing_years = [y for y in target_years if y not in existing_years]
            
            if missing_years:
                # On prend la dernière année disponible pour remplir les suivantes (Forward fill)
                last_year_data = temp_df[temp_df['an'] == max(existing_years)].copy()
                fill_list = [temp_df]
                for y in missing_years:
                    df_fill = last_year_data.copy()
                    df_fill['an'] = y
                    fill_list.append(df_fill)
                temp_df = pd.concat(fill_list, ignore_index=True)
        # Note : Si c'est 'benef_aah' et qu'il a des années, on ne fait rien, il restera tel quel.

    # B. NORMALISATION ÉCHELLE (Département / Commune)
    if is_dept:
        temp_df['code_dept'] = temp_df['codgeo'].str.zfill(2)
        for col in get_variable_cols(temp_df):
            if not col.endswith('_dept'): # Éviter de renommer deux fois
                temp_df = temp_df.rename(columns={col: f"{col}_dept"})
    else:
        temp_df['code_dept'] = temp_df['codgeo'].str[:2]
        
    processed_dfs.append(temp_df)

# Les sections 3, 4 et 5 de ton code restent identiques

# --- 3. PRÉPARATION DU SOCLE GÉOGRAPHIQUE COMPLET ---

# On crée une base "squelette" avec toutes les communes x toutes les années
annees_projet = list(range(2016, 2024))
squelette_list = []

for annee in annees_projet:
    temp_geo = geo.copy()
    temp_geo['an'] = annee
    squelette_list.append(temp_geo)

# Notre base merged commence avec toutes les communes de France pour toutes les années
merged = pd.concat(squelette_list, ignore_index=True)
merged['code_dept'] = merged['codgeo'].str[:2] # Clé pour les futurs merges Dept

print(f"Socle complet créé : {merged.shape[0]} lignes (Communes: {geo.shape[0]} x Années: {len(annees_projet)})")

# --- 4. CONSTRUCTION DE LA BASE FINALE PAR MERGE ---

for i, next_df in enumerate(processed_dfs): # On commence bien à 0 maintenant
    is_dept = next_df['codgeo'].str.len().max() <= 3
    
    if is_dept:
        keys = ['code_dept', 'an']
        cols_to_use = keys + get_variable_cols(next_df)
        # On utilise drop_duplicates pour être sûr de ne pas multiplier les lignes du squelette
        merged = merged.merge(next_df[cols_to_use].drop_duplicates(keys), 
                             on=keys, how='left')
    else:
        keys = ['codgeo', 'an']
        vars_to_add = get_variable_cols(next_df)
        cols_to_use = keys + vars_to_add
        
        # Merge left obligatoire pour ne pas perdre les communes du squelette qui n'ont pas de données
        merged = merged.merge(next_df[cols_to_use], on=keys, how='left', suffixes=('', '_drop'))
        
        # Nettoyage des doublons de colonnes (libgeo, etc.)
        cols_to_drop = [c for c in merged.columns if c.endswith('_drop')]
        merged = merged.drop(columns=cols_to_drop)

    print(f"  + Base {i+1} ({next_df.shape[1]-2} variables) ajoutée → {merged.shape}")

# --- 5. RÉORGANISATION ET TRI ---

# On définit les colonnes d'identification au sens large
# (Celles venant de geo_ + nos clés de merge)
id_cols = ['codgeo', 'an'] 
# On récupère toutes les colonnes qui commencent par 'geo_'
geo_info_cols = [c for c in merged.columns if c.startswith('geo_')]
# Les variables prédictives (tout le reste)
feature_cols = [c for c in merged.columns if c not in id_cols + geo_info_cols]

# Réorganisation de l'ordre des colonnes : IDs, puis Infos Geo, puis Variables
full_column_order = id_cols + geo_info_cols + feature_cols
merged = merged[full_column_order]

# Tri par commune et par année pour avoir un beau panel
merged = merged.sort_values(['codgeo', 'an']).reset_index(drop=True)

print(f"Base finale réorganisée : {merged.shape[0]} lignes × {merged.shape[1]} colonnes")

Expansion spécifique pour codefrr sur [2024, 2025]
Expansion spécifique pour zonage_afr22 sur [2022, 2023, 2024, 2025]
Expansion spécifique pour aav2020 sur [2020, 2021, 2022, 2023, 2024, 2025]
Expansion spécifique pour ti_zpp2 sur [2023, 2024, 2025]
Expansion spécifique pour pvd_zpp sur [2020, 2021, 2022, 2023, 2024, 2025]
Expansion spécifique pour va_zpp sur [2023, 2024, 2025]
Socle complet créé : 279920 lignes (Communes: 34990 x Années: 8)
  + Base 1 (3 variables) ajoutée → (279920, 20)
  + Base 2 (3 variables) ajoutée → (279920, 21)
  + Base 3 (3 variables) ajoutée → (279920, 22)
  + Base 4 (3 variables) ajoutée → (279920, 23)
  + Base 5 (3 variables) ajoutée → (279920, 24)
  + Base 6 (3 variables) ajoutée → (279920, 25)
  + Base 7 (4 variables) ajoutée → (279920, 27)
  + Base 8 (3 variables) ajoutée → (279920, 28)
  + Base 9 (3 variables) ajoutée → (279920, 29)
  + Base 10 (3 variables) ajoutée → (279920, 30)
  + Base 11 (3 variables) ajoutée → (279920, 31)
  + Base 12 (3 variable

#### 2.5 Résumé de la base intermédiaire contenant les caractéristiques socio-économiques des communes (les features)

In [7]:
print("\n=== RÉSUMÉ DE LA BASE DE DONNÉES FINALE ===")
print(f"Nombre de communes uniques : {merged['codgeo'].nunique()}")
print(f"Années couvertes           : {sorted(merged['an'].unique())}")
print(f"Nombre total d'observations : {len(merged)}")

print(f"\n{'Variable':<45} {'Non-nuls':>10} {'Couverture':>12}")
print("-" * 72)

# On affiche le taux de remplissage pour les variables prédictives uniquement
for col in feature_cols:
    n_non_null = merged[col].notna().sum()
    pct = 100 * n_non_null / len(merged)
    # On affiche en rouge ou avec une alerte si la couverture est < 50%
    alert = "!!" if pct < 50 else " "
    print(f"{alert} {col:<43} {n_non_null:>10} {pct:>11.1f}%")

print("-" * 72)


=== RÉSUMÉ DE LA BASE DE DONNÉES FINALE ===
Nombre de communes uniques : 34990
Années couvertes           : [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Nombre total d'observations : 279920

Variable                                        Non-nuls   Couverture
------------------------------------------------------------------------
  code_dept                                       279920       100.0%
!! codefrr                                              0         0.0%
!! benef_aah                                        16809         6.0%
!! part_alf                                        137960        49.3%
!! nb_als                                          139172        49.7%
!! part_apl                                        137960        49.3%
!! zonage_afr22                                     69728        24.9%
!! aav2020                                         103884        37.1%
!! aav2020_lib

In [8]:
# --- BILAN DE COUVERTURE TEMPORELLE ---

# 1. On liste les variables à analyser (on exclut les IDs et les infos géo fixes)
id_and_geo = ['codgeo', 'an', 'code_dept'] + [c for c in merged.columns if c.startswith('geo_')]
vars_to_analyze = [c for c in merged.columns if c not in id_and_geo]

# 2. Calcul du taux de remplissage par (Variable x Année)
# On crée une table pivot où chaque cellule est le % de non-nuls
coverage_pivot = merged.groupby('an')[vars_to_analyze].apply(lambda x: (x.notna().sum() / len(x)) * 100)

# 3. Transposer pour avoir les variables en lignes et les années en colonnes
coverage_dashboard = coverage_pivot.T

# 4. Affichage stylisé (optionnel : utilise seaborn pour une heatmap visuelle)
print("=== TAUX DE COUVERTURE PAR VARIABLE ET PAR ANNÉE (%) ===")
# On arrondit pour la lisibilité
display(coverage_dashboard.round(1))

=== TAUX DE COUVERTURE PAR VARIABLE ET PAR ANNÉE (%) ===


an,2016,2017,2018,2019,2020,2021,2022,2023
codefrr,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
benef_aah,0.0,0.0,0.0,0.0,0.0,48.0,0.0,0.0
part_alf,0.0,0.0,0.0,0.0,98.6,98.6,98.6,98.5
nb_als,0.0,0.0,0.0,0.0,99.5,99.4,99.4,99.4
part_apl,0.0,0.0,0.0,0.0,98.6,98.6,98.6,98.5
zonage_afr22,0.0,0.0,0.0,0.0,0.0,0.0,99.6,99.6
aav2020,0.0,0.0,0.0,0.0,74.2,74.2,74.2,74.2
aav2020_libgeo,0.0,0.0,0.0,0.0,74.2,74.2,74.2,74.2
benef_apa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ti_zpp2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,99.6


In [9]:
# --- RÉSUMÉ DES DISPONIBILITÉS TEMPORELLES ---

# 1. On récupère la liste des variables (hors IDs et Geo)
id_and_geo = ['codgeo', 'an', 'code_dept'] + [c for c in merged.columns if c.startswith('geo_')]
vars_to_analyze = [c for c in merged.columns if c not in id_and_geo]

print(f"{'Variable':<45} | {'Années présentes'}")
print("-" * 80)

for col in vars_to_analyze:
    # On récupère les années où la colonne n'est pas vide (NaN)
    annees_dispos = merged[merged[col].notna()]['an'].unique()
    
    if len(annees_dispos) == 0:
        annees_str = "⚠️ AUCUNE DONNÉE"
    else:
        # On trie les années et on les transforme en chaîne
        annees_triées = sorted(annees_dispos)
        
        # Si c'est une suite continue (ex: 2016, 2017, 2018...), on simplifie l'affichage
        if len(annees_triées) > 2 and (annees_triées[-1] - annees_triées[0] == len(annees_triées) - 1):
            annees_str = f"{annees_triées[0]} à {annees_triées[-1]}"
        else:
            annees_str = ", ".join(map(str, annees_triées))
            
    print(f"{col:<45} | {annees_str}")

Variable                                      | Années présentes
--------------------------------------------------------------------------------
codefrr                                       | ⚠️ AUCUNE DONNÉE
benef_aah                                     | 2021
part_alf                                      | 2020 à 2023
nb_als                                        | 2020 à 2023
part_apl                                      | 2020 à 2023
zonage_afr22                                  | 2022, 2023
aav2020                                       | 2020 à 2023
aav2020_libgeo                                | 2020 à 2023
benef_apa                                     | ⚠️ AUCUNE DONNÉE
ti_zpp2                                       | 2023
part_de_25                                    | ⚠️ AUCUNE DONNÉE
dens_pop                                      | 2016, 2022
ind_vieillist                                 | 2016, 2022
part_vacant_plus_2ans                         | ⚠️ AUCUNE DONNÉE
med_disp   

In [10]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 279920 entries, 0 to 279919
Data columns (total 52 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   codgeo                 279920 non-null  object 
 1   an                     279920 non-null  int64  
 2   geo_typecom            279920 non-null  object 
 3   geo_reg_code           279920 non-null  int64  
 4   geo_reg_nom            279920 non-null  object 
 5   geo_dep_code           279920 non-null  object 
 6   geo_dep_nom            279920 non-null  object 
 7   geo_epci_code          279920 non-null  object 
 8   geo_epci_nom           279560 non-null  object 
 9   geo_code_postal        279920 non-null  int64  
 10  geo_zone_emploi        279560 non-null  float64
 11  geo_population         279920 non-null  int64  
 12  geo_superficie_km2     279920 non-null  int64  
 13  geo_densite            279784 non-null  float64
 14  geo_altitude_moyenne   279920 non-nu

## 3. Données de valeurs foncières (DVF) (2017–2023)

Format : une ligne par commune × année. Clé de merge : `codgeo` + `an`.

In [11]:
# --- 1. CHARGEMENT ---
dvf_path = os.path.join(base_path, "Indicateurs-immobiliers")
dvf_frames = []
for file in sorted(os.listdir(dvf_path)):
    if file.startswith("dvf") and file.endswith(".csv"):
        # On peut aussi spécifier le séparateur si c'est du CSV français (souvent ';')
        df = pd.read_csv(os.path.join(dvf_path, file), low_memory=False)
        dvf_frames.append(df)

dvf = pd.concat(dvf_frames, ignore_index=True)

# Nettoyage des clés
dvf = dvf.rename(columns={'INSEE_COM': 'codgeo', 'Annee': 'an'})
dvf['codgeo'] = dvf['codgeo'].astype(str).str.zfill(5)
dvf['an'] = dvf['an'].astype(int)

# --- 2. AGRÉGATION ---
# On vérifie quelles colonnes sont des variables (prix au m2, nombre de ventes, etc.)
dvf_vars = [c for c in dvf.columns if c not in ['codgeo', 'an', 'LIBCOM']]

# On groupe par commune et année pour avoir une seule ligne
# On prend la moyenne pour les prix, et la somme pour les volumes de ventes
# (Adapte les fonctions selon tes colonnes réelles)
dvf_grouped = dvf.groupby(['codgeo', 'an'])[dvf_vars].mean().reset_index()

print(f"DVF agrégé (1 ligne par commune/an) : {dvf_grouped.shape}")

# Avant le merge, on préfixe les colonnes de données (sauf les clés)
cols_to_prefix = [c for c in dvf_grouped.columns if c not in ['codgeo', 'an']]
dvf_grouped = dvf_grouped.rename(columns={c: f"dvf_{c}" for c in cols_to_prefix})

# Maintenant on merge
merged = merged.merge(dvf_grouped, on=['codgeo', 'an'], how='left')

print(f"Merge DVF terminé. Nouvelle taille de la base : {merged.shape}")

DVF agrégé (1 ligne par commune/an) : (206489, 11)
Merge DVF terminé. Nouvelle taille de la base : (279920, 61)


In [12]:
# Nombre de communes couvertes par le DVF chaque année
repartition_annee = dvf.groupby('an')['codgeo'].count().reset_index()
repartition_annee.columns = ['Année', 'Nombre de lignes (transactions/communes)']
print("=== COUVERTURE TEMPORELLE DVF ===")
display(repartition_annee)

=== COUVERTURE TEMPORELLE DVF ===


,Année,Nombre de lignes (transactions/communes)
0,2017,26687
1,2018,27074
2,2019,30231
3,2020,30470
4,2021,31101
5,2022,30915
6,2023,30011


## 4. Les données de dotations aux communes
Format **long**  : `Exercice` + `Nom Collectivité` + `Variable`

On pivote sur `Variable` pour avoir une colonne par type de dotation. La clé de jointure est le `codgeo` reconstitué depuis le `Nom Collectivité` — ou directement depuis une colonne code commune si elle existe dans le fichier.

In [13]:
# Chargement du fichier dotations-communes.csv
aides_path = os.path.join(comptes_path, "dotations-communes.csv")
df_aides = pd.read_csv(aides_path, sep=';', encoding='utf-8')

print(f"Nombre de colonnes : {len(df_aides.columns)}")
print("-" * 30)
print(df_aides.columns.tolist())


/var/folders/hv/h08fhnrs21dcljcln68y63bm0000gn/T/ipykernel_41239/1855873015.py:3: DtypeWarning: Columns (5,7,9,11) have mixed types. Specify dtype option on import or set low_memory=False.
  df_aides = pd.read_csv(aides_path, sep=';', encoding='utf-8')


Nombre de colonnes : 17
------------------------------
['Exercice', 'Strate population', 'Champ géographique', 'Code Région', 'Nom Région', 'Code Département', 'Nom Département', 'Code Siren Groupement à fiscalité propre', 'Nom Groupement à fiscalité propre', 'Code Insee Chef-lieu', 'Nom Chef-lieu', 'Code Insee Commune', 'Nom Collectivité', 'Catégorie', 'Unité', 'Variable', 'Valeur']


In [14]:
# On renomme pour correspondre à ton socle 'merged'
df_aides = df_aides.rename(columns={'Code Insee Commune': 'codgeo', 'Exercice': 'an'})

# Harmonisation des types
df_aides['codgeo'] = df_aides['codgeo'].astype(str).str.strip().str.zfill(5)
df_aides['an'] = df_aides['an'].astype(int)

# --- 2. LE PIVOT (Passage du format Long au format Wide) ---
# On ne garde que les colonnes essentielles pour le pivot
# 'Variable' deviendra les noms de colonnes, 'Valeur' le contenu
df_dot_pivot = df_aides.pivot_table(
    index=['codgeo', 'an'], 
    columns='Variable', 
    values='Valeur', 
    aggfunc='sum' # Au cas où il y aurait des doublons
).reset_index()

# Optionnel : Ajouter un préfixe pour identifier ces variables dans ton ML
# Exemple : 'DGF' devient 'dot_DGF'
dot_cols = [c for c in df_dot_pivot.columns if c not in ['codgeo', 'an']]
df_dot_pivot = df_dot_pivot.rename(columns={c: f"dot_{c.replace(' ', '_')}" for c in dot_cols})

print(f"Base dotations pivotée : {df_dot_pivot.shape[0]} lignes et {df_dot_pivot.shape[1]-2} nouvelles variables.")
print(df_dot_pivot.columns.tolist())

Base dotations pivotée : 32786 lignes et 261 nouvelles variables.
['codgeo', 'an', 'dot_Attribution_compensant_le_transfert_de_la_part_CPS_aux_EPCI_à_FA', 'dot_Attribution_de_compensation_de_la_commune', 'dot_Attribution_de_compensation_pour_nuisances_environnementales_de_la_commune', "dot_Bases_brutes_de_CFE_de_l'EPCI_(hors_ZAE/ZE)", "dot_Bases_brutes_de_CFE_de_l'EPCI_sur_ZAE/ZE", 'dot_Bases_brutes_de_CFE_de_la_commune', 'dot_Bases_brutes_de_FB', 'dot_Bases_brutes_de_FNB', 'dot_Bases_brutes_de_TH', "dot_Bases_brutes_de_TH_totales_de_l'EPCI_(FPU)", "dot_Bases_brutes_de_THRP_2020_de_l'EPCI", 'dot_Bases_brutes_de_THRP_2020_de_la_commune', 'dot_Bases_brutes_de_THRS', "dot_Bases_brutes_de_THRS_totales_de_l'EPCI_(FPU)", "dot_Bases_de_CFE_locaux_industriels_exonérées_de_l'EPCI_(hors_ZAE/ZE)", "dot_Bases_de_CFE_locaux_industriels_exonérées_de_l'EPCI_sur_ZAE/ZE", 'dot_Bases_de_CFE_locaux_industriels_exonérées_de_la_commune', 'dot_Bases_de_FB_locaux_industriels_exonérées_de_la_commune', 'dot_Ba

In [15]:
# --- BILAN SIMPLIFIÉ DE COUVERTURE DES DOTATIONS (CORRIGÉ) ---

# 1. On récupère la liste des colonnes qui commencent par 'dot_'
# Ce sont les variables que nous avons pivotées et renommées
dot_vars_prefixed = [c for c in df_dot_pivot.columns if c.startswith('dot_')]

# 2. Calcul du taux de remplissage global (% de valeurs non-nulles)
stats_couverture = df_dot_pivot[dot_vars_prefixed].notna().mean() * 100

# 3. Fonction pour compter le nombre d'années uniques par variable
def count_years(col_name):
    # On filtre les lignes où la variable n'est pas NaN, puis on compte les années uniques
    return df_dot_pivot.loc[df_dot_pivot[col_name].notna(), 'an'].nunique()

# On applique la fonction sur chaque colonne dot_
annees_presence = pd.Series({col: count_years(col) for col in dot_vars_prefixed})

# 4. Création du tableau de bord final
bilan_dotations = pd.DataFrame({
    'Taux Remplissage (%)': stats_couverture,
    'Nombre d\'années': annees_presence
}).sort_values('Taux Remplissage (%)', ascending=False)

print(f"=== BILAN SUR {len(dot_vars_prefixed)} VARIABLES DE DOTATIONS ===")
display(bilan_dotations.round(1))

=== BILAN SUR 261 VARIABLES DE DOTATIONS ===


,Taux Remplissage (%),Nombre d'années
dot_Montant_Dotation_forfaitaire,100.0,8
dot_Montant_Dotation_DGF,100.0,8
dot_Superficie,100.0,8
dot_Population_INSEE,100.0,8
dot_Population_DGF,100.0,8
...,...,...
dot_Quote-part_DNP_-_Part_capacité_financière,0.0,1
dot_Quote-part_DSU/DSR_-_Part_capacité_financière,0.0,1
dot_Dotation_de_consolidation_pour_les_CN_sur_périmètre_N,0.0,2
dot_Quote-part_DNP_-_Part_superficie,0.0,1


In [16]:
# On utilise 'left' pour ne pas perdre les communes qui n'auraient pas reçu de dotations
merged = merged.merge(df_dot_pivot, on=['codgeo', 'an'], how='left')

# Remplissage des NaNs par 0 (si pas de dotation listée = 0 euro reçu)
new_dot_cols = [c for c in merged.columns if c.startswith('dot_')]
merged[new_dot_cols] = merged[new_dot_cols].fillna(0)

print(f"Dotations fusionnées ! Taille de la base finale : {merged.shape}")
merged.head()

Dotations fusionnées ! Taille de la base finale : (279920, 322)


,codgeo,an,geo_typecom,geo_reg_code,geo_reg_nom,geo_dep_code,geo_dep_nom,geo_epci_code,geo_epci_nom,geo_code_postal,...,dot_Taux_cumulé_de_CFE,dot_Taux_moyen_pondéré_N,dot_Taux_moyen_pondéré_N_strate,dot_Taux_moyen_pondéré_N-1,dot_Taux_moyen_pondéré_N-1_strate,dot_Taxe_additionnelle_sur_les_installations_nucélaires_de_base,dot_Taxe_locale_sur_la_publicité_extérieure,dot_Taxe_sur_les_jeux_EPCI,dot_Taxe_sur_les_pylônes_électriques,dot_Valeur_de_l'indice_synthétique_de_classement_de_la_commune_à_la_DSU
0,01001,2016,COM,84,Auvergne-Rhône-Alpes,01,Ain,200069193,CC de la Dombes,1400,...,0.0000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.00,0.0,0.0
1,01001,2017,COM,84,Auvergne-Rhône-Alpes,01,Ain,200069193,CC de la Dombes,1400,...,0.0000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.00,0.0,0.0
2,01001,2018,COM,84,Auvergne-Rhône-Alpes,01,Ain,200069193,CC de la Dombes,1400,...,0.2366,0.186013,0.213744,0.185067,0.211305,0.0,0.0,0.00,0.0,0.0
3,01001,2019,COM,84,Auvergne-Rhône-Alpes,01,Ain,200069193,CC de la Dombes,1400,...,0.2364,0.184712,0.215325,0.186013,0.214303,0.0,0.0,1407.69,0.0,0.0
4,01001,2020,COM,84,Auvergne-Rhône-Alpes,01,Ain,200069193,CC de la Dombes,1400,...,0.2361,0.183455,0.215797,0.184712,0.215355,0.0,0.0,0.00,0.0,0.0


## 5. Données des comptes des communes (2017-2023) : les targets
Format **long** : une ligne par commune × année × agrégat (54 agrégats disponibles, ex: Frais de personnel, Épargne brute, Dépenses d'équipement...).

On pivote vers le format **large** (une ligne par commune × année, une colonne par agrégat) avant de merger.

In [17]:
path_detail_comptes = os.path.join(comptes_path, "ofgl-base-communes.csv")
df_comptes = pd.read_csv(path_detail_comptes, sep=';', low_memory=False)      

print(f"Colonnes OFGL : {df_comptes.columns.tolist()}")
print(f"Dimensions : {df_comptes.shape}")
print(df_comptes.columns.tolist())
df_comptes.head(3)

Colonnes OFGL : ['Exercice', 'Outre-mer', 'Code Insee 2024 Région', 'Nom 2024 Région', 'Code Insee 2024 Département', 'Nom 2024 Département', 'Code Siren 2024 EPCI', 'Nom 2024 EPCI', 'Strate population 2024', 'Commune rurale', 'Commune de montagne', 'Commune touristique', 'Tranche revenu par habitant', 'Présence QPV', 'Code Insee 2024 Commune', 'Nom 2024 Commune', 'Catégorie', 'Code Siren Collectivité', 'Code Insee Collectivité', 'Siret Budget', 'Libellé Budget', 'Type de budget', 'Nomenclature', 'Agrégat', 'Montant', 'Montant en millions', 'Population totale', 'Montant en € par habitant', 'Compte 2024 Disponible', 'code_type_budget', 'ordre_analyse1_section1', 'ordre_analyse1_section2', 'ordre_analyse1_section3', 'ordre_analyse2_section1', 'ordre_analyse2_section2', 'ordre_analyse2_section3', 'ordre_analyse3_section1', 'ordre_analyse3_section2', 'ordre_analyse3_section3', 'ordre_analyse4_section1', 'annee_join', 'Population totale du dernier exercice']
Dimensions : (22364887, 42)
['Ex

,Exercice,Outre-mer,Code Insee 2024 Région,Nom 2024 Région,Code Insee 2024 Département,Nom 2024 Département,Code Siren 2024 EPCI,Nom 2024 EPCI,Strate population 2024,Commune rurale,...,ordre_analyse1_section3,ordre_analyse2_section1,ordre_analyse2_section2,ordre_analyse2_section3,ordre_analyse3_section1,ordre_analyse3_section2,ordre_analyse3_section3,ordre_analyse4_section1,annee_join,Population totale du dernier exercice
0,2017,Non,84.0,Auvergne-Rhône-Alpes,69,Rhône,246900625.0,Communauté de communes du Pays de l'Arbresle (...,4.0,Non,...,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,2017,2028.0
1,2017,Non,84.0,Auvergne-Rhône-Alpes,69,Rhône,246900625.0,Communauté de communes du Pays de l'Arbresle (...,4.0,Non,...,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,2017,2028.0
2,2017,Non,84.0,Auvergne-Rhône-Alpes,69,Rhône,246900740.0,Communauté de communes du Pays Mornantais (COP...,5.0,Non,...,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,2017,4730.0


In [18]:
# --- 1. NETTOYAGE PRÉALABLE ---
df_comptes = df_comptes.rename(columns={
    'Code Insee 2024 Commune': 'codgeo', 
    'Exercice': 'an'
})

df_comptes['codgeo'] = df_comptes['codgeo'].astype(str).str.strip().str.zfill(5)
df_comptes['an'] = df_comptes['an'].astype(int)

# --- 2. LE PIVOT ---
df_comptes_pivot = df_comptes.pivot_table(
    index=['codgeo', 'an'], 
    columns='Agrégat', 
    values='Montant', 
    aggfunc='sum'
).reset_index()

# --- 3. RENOMMAGE (Correction de l'erreur) ---
# On définit agregat_cols ici à partir du résultat du pivot
agregat_cols = [c for c in df_comptes_pivot.columns if c not in ['codgeo', 'an']]

new_names = {}
for c in agregat_cols:
    clean_name = c.replace(' ', '_').replace("'", "_")
    new_names[c] = f"ofgl_{clean_name}"

df_comptes_pivot = df_comptes_pivot.rename(columns=new_names)

print(f"✅ Colonnes renommées. Aperçu : {df_comptes_pivot.columns[:5].tolist()}")
print(f"✅ Base OFGL pivotée : {df_comptes_pivot.shape[0]} lignes et {len(agregat_cols)} agrégats en colonnes.")
print(df_comptes_pivot.columns.tolist())

✅ Colonnes renommées. Aperçu : ['codgeo', 'an', 'ofgl_Achats_et_charges_externes', 'ofgl_Annuité_de_la_dette', 'ofgl_Autres_dotations_de_fonctionnement']
✅ Base OFGL pivotée : 279453 lignes et 54 agrégats en colonnes.
['codgeo', 'an', 'ofgl_Achats_et_charges_externes', 'ofgl_Annuité_de_la_dette', 'ofgl_Autres_dotations_de_fonctionnement', 'ofgl_Autres_dotations_et_subventions', 'ofgl_Autres_dépenses_d_investissement', 'ofgl_Autres_dépenses_de_fonctionnement', 'ofgl_Autres_impôts_et_taxes', 'ofgl_Autres_recettes_d_investissement', 'ofgl_Autres_recettes_de_fonctionnement', 'ofgl_Capacité_ou_besoin_de_financement', 'ofgl_Charges_financières', 'ofgl_Concours_de_l_Etat', 'ofgl_Crédits_de_trésorerie', 'ofgl_DETR', 'ofgl_Dotation_globale_de_fonctionnement', 'ofgl_Dépenses_d_intervention', 'ofgl_Dépenses_d_investissement', 'ofgl_Dépenses_d_investissement_hors_remb', 'ofgl_Dépenses_d_équipement', 'ofgl_Dépenses_de_fonctionnement', 'ofgl_Dépenses_totales', 'ofgl_Dépenses_totales_hors_remb', 'ofg

In [19]:
# --- 3. MERGE FINAL ---
# On merge directement sur 'merged' (pas merged_final) pour tout consolider au même endroit
merged = merged.merge(df_comptes_pivot, on=['codgeo', 'an'], how='left')

# Vérification du résultat
print(f"✅ DATASET FINAL PRÊT : {merged.shape[0]} lignes × {merged.shape[1]} colonnes")

✅ DATASET FINAL PRÊT : 279920 lignes × 376 colonnes


## 6. Export

In [20]:
# --- RÉORGANISATION ET EXPORT ---
# On travaille sur 'merged' qui contient maintenant TOUTES les bases
# (geo, ANCT, dot, ofgl). Le DVF sera ajouté ici quand disponible.

# 1. Colonnes d'identification
id_cols = ['codgeo', 'an']
if 'libgeo' in merged.columns:
    id_cols = ['codgeo', 'libgeo', 'an']

# 2. Regroupement par préfixe — on relit depuis merged.columns (source de vérité)
geo_cols  = [c for c in merged.columns if c.startswith('geo_')]
if 'code_dept' in merged.columns:
    geo_cols = ['code_dept'] + geo_cols

dvf_cols  = [c for c in merged.columns if c.startswith('dvf_')]
ofgl_cols = [c for c in merged.columns if c.startswith('ofgl_')]
dot_cols  = [c for c in merged.columns if c.startswith('dot_')]
fisc_cols = [c for c in merged.columns if c.startswith('fisc_')]

# 3. Colonnes ANCT (tout ce qui reste sans préfixe)
classified = set(id_cols + geo_cols + dvf_cols + ofgl_cols + dot_cols + fisc_cols)
anct_cols  = [c for c in merged.columns if c not in classified]

# 4. Ordre final
col_order = id_cols + geo_cols + anct_cols + dvf_cols + fisc_cols + dot_cols + ofgl_cols
col_order = [c for c in col_order if c in merged.columns]  # sécurité
merged = merged[col_order].sort_values(['codgeo', 'an']).reset_index(drop=True)

# 5. Bilan par bloc
print('=== Base finale complète ===')
print(f'Dimensions      : {merged.shape[0]} lignes × {merged.shape[1]} colonnes')
print(f'Communes uniques: {merged["codgeo"].nunique()}')
print(f'Années couvertes: {sorted(merged["an"].unique())}')
print()
blocs = {
    'ANCT Observatoire': anct_cols,
    'Référentiel géo'  : geo_cols,
    'DVF'              : dvf_cols,
    'Fiscalité délib.' : fisc_cols,
    'Dotations'        : dot_cols,
    'OFGL comptes'     : ofgl_cols,
}
print(f"{'Bloc':<22} {'Colonnes':>9} {'Remplissage moyen':>20}")
print('-' * 55)
for nom, cols in blocs.items():
    nb = len(cols)
    taux = merged[cols].notna().mean().mean() if cols else float('nan')
    taux_str = f'{taux:.1%}' if cols else 'N/A (pas encore mergé)'
    print(f'  {nom:<20} {nb:>9} {taux_str:>20}')
print()
print(f"Taux de remplissage global : {merged.notna().mean().mean():.1%}")


=== Base finale complète ===
Dimensions      : 279920 lignes × 376 colonnes
Communes uniques: 34990
Années couvertes: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]

Bloc                    Colonnes    Remplissage moyen
-------------------------------------------------------
  ANCT Observatoire           33                26.8%
  Référentiel géo             17               100.0%
  DVF                          9                73.7%
  Fiscalité délib.             0 N/A (pas encore mergé)
  Dotations                  261               100.0%
  OFGL comptes                54                78.5%

Taux de remplissage global : 89.9%


In [21]:

# 6. Export
output_parquet = os.path.join(base_path, 'base_finale_communes.parquet')
merged.to_parquet(output_parquet, engine='pyarrow', compression='snappy', index=False)
print(f"\n✓ Parquet exporté : {output_parquet}")
print(f"  Poids : {os.path.getsize(output_parquet) / (1024**2):.1f} Mo")



✓ Parquet exporté : /Users/margo/Documents/Margot/0.Polytechnique/Statistical-machine-learning/Projet-final/Data/base_finale_communes.parquet
  Poids : 89.8 Mo


In [23]:
#Export en parquet du dataset OFGL

# 1. Définir le chemin d'export (adapte 'comptes_path' selon ton arborescence)
export_path = os.path.join(comptes_path, "ofgl_comptes_pivot.parquet")

# 2. Exporter le DataFrame pivoté
# On utilise la compression 'snappy' pour un compromis idéal vitesse/taille
df_comptes_pivot.to_parquet(
    export_path, 
    engine='pyarrow', 
    compression='snappy',
    index=False # Évite de créer une colonne d'index inutile dans le fichier
)

print(f"✅ Dataset OFGL exporté avec succès : {export_path}")
print(f"Poids du fichier : {os.path.getsize(export_path) / (1024**2):.2f} Mo")

✅ Dataset OFGL exporté avec succès : /Users/margo/Documents/Margot/0.Polytechnique/Statistical-machine-learning/Projet-final/Data/Comptes-communes/ofgl_comptes_pivot.parquet
Poids du fichier : 80.43 Mo


In [22]:

output_csv = os.path.join(base_path, 'base_finale_communes.csv')
merged.to_csv(output_csv, index=False, encoding='utf-8-sig')
print(f"✓ CSV exporté     : {output_csv}")
print(f"  Poids : {os.path.getsize(output_csv) / (1024**2):.1f} Mo")

✓ CSV exporté     : /Users/margo/Documents/Margot/0.Polytechnique/Statistical-machine-learning/Projet-final/Data/base_finale_communes.csv
  Poids : 470.6 Mo
